# ECON 662 — Problem Set 2
## Nested Fixed Point (NFXP) Estimation — Harold Zurcher Jr.

**Problem:** Harold Jr. manages 1,000 buses. Each period $t$ he observes
mileage $x_{it} \in \{1,\ldots,7\}$ and replacement cost $RC_t$, then decides
whether to replace the engine ($a=1$) or maintain ($a=0$).

| Parameter | Description |
|-----------|-------------|
| $\theta_1$ | Marginal cost of mileage (maintenance cost slope) |
| $\theta_2$ | Scaling of replacement cost in utility |
| $\rho_0$ | AR(1) intercept for $RC_t$ |
| $\rho_1$ | AR(1) persistence for $RC_t$ |
| $\sigma_\rho$ | Std dev of AR(1) innovation |

**Algorithm:** NFXP (Rust 1987) — outer optimizer wraps an inner Bellman fixed-point solver.

---
**Design philosophy of this notebook:**  
Every computation is written as an explicit matrix operation so you can see exactly what is happening at each step. No black-box wrappers — only `numpy`, `scipy.stats.norm`, and `scipy.optimize.minimize`.

## 0. Import Libraries

In [62]:
import numpy as np
import pandas as pd
from scipy.stats import norm        # norm.cdf, norm.logpdf — used explicitly
from scipy.optimize import minimize # outer Nelder-Mead only

# No other packages. Everything else is plain numpy matrix algebra.

## 1. Load Data

In [63]:
# Columns: i (bus id), t (month 1-100), a (action: 0=maintain, 1=replace),
#          x (mileage state: 1-7), rc (replacement cost, continuous)
df = pd.read_csv("ddc_pset.csv")

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
df.head(10)

Loaded 100,000 observations for 1,000 buses.
RC range: [31.5000, 62.5000]


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


## 2. Global Constants

In [64]:
BETA        = 0.95           # discount factor (given in problem)
N_X         = 7              # number of mileage states {1, 2, ..., 7}
N_RC        = 25             # number of grid points for discretising RC_t
GAMMA_EULER = 0.5772156649   # Euler-Mascheroni constant (mean of Type I EV dist.)

## 3. Pre-compute RC Lag Pairs for the AR(1) Log-Likelihood

To avoid re-sorting and grouping inside every call to the objective function,
we build $(RC_{t-1}, RC_t)$ pairs **once** here.
For each bus $i$ we pair $(RC_{t-1}, RC_t)$ for $t = 2, \ldots, 100$.

In [65]:
# Sort by bus then time to guarantee consecutive ordering
df = df.sort_values(["i", "t"]).reset_index(drop=True)

rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    rc_vals = g["rc"].values
    rc_prev_list.append(rc_vals[:-1])   # RC_{t-1}
    rc_curr_list.append(rc_vals[1:])    # RC_t

rc_prev_vec = np.concatenate(rc_prev_list)   # shape (99_000,)
rc_curr_vec = np.concatenate(rc_curr_list)   # shape (99_000,)

print(f"AR(1) pairs constructed: {len(rc_curr_vec):,} observations.")

AR(1) pairs constructed: 99,000 observations.


## 4. OLS Warm-Start for AR(1) Parameters

The AR(1) log-likelihood is separable from the choice likelihood,
so OLS gives **consistent starting values** for $(\rho_0, \rho_1, \sigma_\rho)$:

$$RC_t = \rho_0 + \rho_1 \cdot RC_{t-1} + \varepsilon_t$$

In matrix form: $\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \mathbf{e}$
where $\mathbf{X} = [\mathbf{1} \mid RC_{t-1}]$ and $\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top \mathbf{y}$.

In [66]:
# Design matrix X = [1,  RC_{t-1}]   shape: (99000, 2)
X = np.column_stack([np.ones(len(rc_prev_vec)), rc_prev_vec])

print(f"Design matrix X shape: {X.shape}")
print("First 5 rows of X:")
print(X[:5])

# X'X  — (2 x 2) matrix
XtX = X.T @ X
print(f"\nX'X =\n{XtX}")

# X'y  — (2,) vector
Xty = X.T @ rc_curr_vec
print(f"\nX'y = {Xty}")

# OLS: β = (X'X)^{-1} X'y   — solve the 2x2 linear system
beta_ols = np.linalg.solve(XtX, Xty)
rho0_ols, rho1_ols = beta_ols

# Residuals and sigma
resid        = rc_curr_vec - X @ beta_ols
sigma_rho_ols = resid.std(ddof=1)

print("\n--- OLS AR(1) Warm-Start ---")
print(f"  rho0      = {rho0_ols:.4f}")
print(f"  rho1      = {rho1_ols:.4f}")
print(f"  sigma_rho = {sigma_rho_ols:.4f}")

Design matrix X shape: (99000, 2)
First 5 rows of X:
[[ 1. 45.]
 [ 1. 49.]
 [ 1. 54.]
 [ 1. 47.]
 [ 1. 56.]]

X'X =
[[9.9000000e+04 4.7025000e+06]
 [4.7025000e+06 2.2682125e+08]]

X'y = [4.7035000e+06 2.2557375e+08]

--- OLS AR(1) Warm-Start ---
  rho0      = 17.8269
  rho1      = 0.6249
  sigma_rho = 4.6060


## 5. Tauchen (1986) Method — Discretise the AR(1) Process for RC

The continuous AR(1) is approximated by a finite-state Markov chain on $N_{RC}$ grid points.

**Steps:**
1. Compute stationary mean $\mu_{ss} = \rho_0/(1-\rho_1)$ and std $\sigma_{ss} = \sigma_\rho / \sqrt{1-\rho_1^2}$
2. Build an evenly-spaced grid over $[\mu_{ss} - m\sigma_{ss},\ \mu_{ss} + m\sigma_{ss}]$
3. Fill $\Pi$ by integrating $\mathcal{N}(\rho_0 + \rho_1 rc_j,\ \sigma_\rho^2)$ over bins around each grid point

$$\Pi_{j,j'} = \Phi\!\left(\frac{rc_{j'} + \delta/2 - \mu_j^{cond}}{\sigma_\rho}\right) - \Phi\!\left(\frac{rc_{j'} - \delta/2 - \mu_j^{cond}}{\sigma_\rho}\right)$$

**Key:** We build two $(N \times N)$ matrices and subtract.

In [67]:
def tauchen(rho0, rho1, sigma_rho, N, m=3.0):
    """
    Tauchen (1986) discretisation of RC_t = rho0 + rho1*RC_{t-1} + e_t.

    Returns
    -------
    rc_grid : ndarray (N,)   — grid points
    Pi      : ndarray (N, N) — transition matrix, Pi[j, jp] = P(RC'=rc[jp] | RC=rc[j])
    """
    # (a) Stationary moments of the AR(1) process
    mu_ss    = rho0 / (1.0 - rho1)
    sigma_ss = sigma_rho / np.sqrt(1.0 - rho1**2)

    # (b) Evenly-spaced grid spanning [mu_ss - m*sigma_ss, mu_ss + m*sigma_ss]
    rc_grid = np.linspace(mu_ss - m * sigma_ss, mu_ss + m * sigma_ss, N)
    delta   = rc_grid[1] - rc_grid[0]     # grid step size

    # (c) Transition matrix — fully vectorised, no loops
    #
    #   mu_cond[j] = rho0 + rho1 * rc_grid[j]   <- conditional mean for each row j
    #
    #   Build two (N x N) broadcast matrices:
    #     mu_cond_mat : row j = mu_cond[j]  (same value repeated across columns)
    #     rc_mat      : col j'= rc_grid[j'] (same value repeated across rows)
    #
    mu_cond_mat = (rho0 + rho1 * rc_grid).reshape(N, 1)  # (N, 1) -> broadcasts to (N, N)
    rc_mat      = rc_grid.reshape(1, N)                   # (1, N) -> broadcasts to (N, N)

    # Interior: CDF difference over bin [rc_jp - delta/2, rc_jp + delta/2]
    Pi = (norm.cdf((rc_mat + delta / 2 - mu_cond_mat) / sigma_rho) -
          norm.cdf((rc_mat - delta / 2 - mu_cond_mat) / sigma_rho))

    # Left endpoint correction: absorb all left-tail mass into column 0
    Pi[:, 0]  = norm.cdf((rc_grid[0]  + delta / 2 - (rho0 + rho1 * rc_grid)) / sigma_rho)

    # Right endpoint correction: absorb all right-tail mass into column N-1
    Pi[:, -1] = 1.0 - norm.cdf((rc_grid[-1] - delta / 2 - (rho0 + rho1 * rc_grid)) / sigma_rho)

    return rc_grid, Pi

### Inspect the Tauchen output at the OLS warm-start values

In [68]:
rc_grid, Pi = tauchen(rho0_ols, rho1_ols, sigma_rho_ols, N_RC)

print(f"rc_grid  shape: {rc_grid.shape}")
print(f"rc_grid: {np.round(rc_grid, 2)}")

print(f"\nPi shape: {Pi.shape}")
print("\nPi (full 25 x 25 transition matrix):")
print(np.round(Pi, 4))

# Sanity check: every row must sum to exactly 1
row_sums = Pi.sum(axis=1)
print(f"\nRow sums (all must be 1.0):  min={row_sums.min():.10f}  max={row_sums.max():.10f}")

rc_grid  shape: (25,)
rc_grid: [29.83 31.3  32.78 34.25 35.73 37.2  38.68 40.15 41.63 43.1  44.58 46.05
 47.53 49.   50.48 51.95 53.43 54.9  56.38 57.85 59.33 60.8  62.28 63.75
 65.23]

Pi shape: (25, 25)

Pi (full 25 x 25 transition matrix):
[[1.001e-01 6.820e-02 9.260e-02 1.134e-01 1.256e-01 1.256e-01 1.135e-01
  9.260e-02 6.830e-02 4.550e-02 2.730e-02 1.490e-02 7.300e-03 3.200e-03
  1.300e-03 5.000e-04 2.000e-04 0.000e+00 0.000e+00 0.000e+00 0.000e+00
  0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [6.930e-02 5.350e-02 7.740e-02 1.011e-01 1.193e-01 1.271e-01 1.224e-01
  1.064e-01 8.360e-02 5.930e-02 3.800e-02 2.200e-02 1.150e-02 5.400e-03
  2.300e-03 9.000e-04 3.000e-04 1.000e-04 0.000e+00 0.000e+00 0.000e+00
  0.000e+00 0.000e+00 0.000e+00 0.000e+00]
 [4.630e-02 4.040e-02 6.220e-02 8.660e-02 1.089e-01 1.236e-01 1.268e-01
  1.175e-01 9.840e-02 7.440e-02 5.080e-02 3.130e-02 1.750e-02 8.800e-03
  4.000e-03 1.600e-03 6.000e-04 2.000e-04 1.000e-04 0.000e+00 0.000e+00
  0.000e+00 0.000e+00 0

## 6. Inner Loop — Value Function Iteration (Bellman Equation)

For given $(\theta_1, \theta_2)$ and the RC Markov chain, we iterate until convergence:

$$v_0(x,j) = -\theta_1 x + \beta \sum_{j'} \Pi_{j,j'} EV(\min(x{+}1,7),\ j')$$

$$v_1(x,j) = -\theta_2\, rc_j + \beta \sum_{j'} \Pi_{j,j'} EV(1,\ j')$$

$$EV(x,j) = \log\!\bigl(\exp v_0 + \exp v_1\bigr) + \gamma_E
\quad \text{(Type I EV closed form)}$$

**Matrix key:** the double sum $\sum_{j'} \Pi_{j,j'} EV(x', j')$ over all $(x', j)$ pairs
is just the matrix product

$$\underbrace{EV}_{7 \times 25} \cdot \underbrace{\Pi^\top}_{25 \times 25} = \underbrace{EV\_cont}_{7 \times 25}$$

One line replaces the inner $j$-loop entirely.

In [69]:
def solve_bellman(theta1, theta2, rc_grid, Pi, tol=1e-8, maxiter=2000):
    """
    Value function iteration for the Zurcher model.

    Returns
    -------
    EV      : ndarray (N_X, N_RC)  — integrated value function at fixed point
    v0_grid : ndarray (N_X, N_RC)  — choice-specific value: maintain
    v1_grid : ndarray (N_X, N_RC)  — choice-specific value: replace
    """
    N_rc = len(rc_grid)
    EV   = np.zeros((N_X, N_rc))   # initialise at zero; iteration finds fixed point

    # --- Pre-compute objects that don't change across iterations ---

    # Mileage transition indices (0-indexed)
    x_idx    = np.arange(N_X)                    # [0, 1, 2, 3, 4, 5, 6]
    x_next_0 = np.minimum(x_idx + 1, N_X - 1)   # maintain: increment, cap at 6 (state 7)
    x_next_1 = 0                                  # replace:  reset to 0 (state 1)

    # Flow utility of maintenance: -theta1 * x  (column vector, broadcast over RC)
    # Shape: (N_X, 1) so it broadcasts with (N_X, N_rc)
    u0_base = (-theta1 * (x_idx + 1)).reshape(N_X, 1)   # (7, 1)

    # Flow utility of replacement: -theta2 * rc_grid[j]  (same for all mileage states)
    # Build it directly as an (N_X, N_rc) array to avoid collapsing to one row.
    u1 = np.broadcast_to((-theta2 * rc_grid).reshape(1, N_rc), (N_X, N_rc))

    for iteration in range(maxiter):

        # ---- Continuation values: single matrix multiply ----
        # EV_cont[xi, j] = sum_{jp} Pi[j, jp] * EV[xi, jp]
        #                 = (EV @ Pi.T)[xi, j]
        # Result shape: (N_X, N_rc)
        EV_cont = EV @ Pi.T                              # (7, 25) @ (25, 25) = (7, 25)

        # ---- Choice-specific value functions ----
        # v0[xi, j] = u0_base[xi] + beta * EV_cont[x_next_0[xi], j]
        v0 = u0_base + BETA * EV_cont[x_next_0, :]      # (7, 25)

        # v1[xi, j] = u1[xi, j] + beta * EV_cont[0, j]  (same row for all xi)
        v1 = u1 + np.broadcast_to(EV_cont[x_next_1, :], (N_X, N_rc)) * BETA

        # ---- Log-sum-exp with numerical stability ----
        # Subtract vmax before exp() to prevent overflow
        vmax   = np.maximum(v0, v1)
        EV_new = np.log(np.exp(v0 - vmax) + np.exp(v1 - vmax)) + vmax + GAMMA_EULER

        diff = np.max(np.abs(EV_new - EV))
        EV   = EV_new

        if diff < tol:
            break

    return EV, v0, v1

### Inspect the Bellman output at the warm-start parameters

In [70]:
EV, v0_grid, v1_grid = solve_bellman(1.0, 1.0, rc_grid, Pi)

print(f"EV shape:      {EV.shape}       (rows=mileage states, cols=RC grid points)")
print(f"v0_grid shape: {v0_grid.shape}")
print(f"v1_grid shape: {v1_grid.shape}")

print("\n--- EV (integrated value function) ---")
print("      rc_grid:  ", np.round(rc_grid[:5], 1), "...")
print(f"x=1 (row 0): {np.round(EV[0, :5], 3)} ...")
print(f"x=4 (row 3): {np.round(EV[3, :5], 3)} ...")
print(f"x=7 (row 6): {np.round(EV[6, :5], 3)} ...")

print("\n--- v0_grid (value of MAINTAINING) ---")
print(f"x=1 (row 0): {np.round(v0_grid[0, :5], 3)} ...")
print(f"x=7 (row 6): {np.round(v0_grid[6, :5], 3)} ...")

print("\n--- v1_grid (value of REPLACING) ---")
print(f"x=1 (row 0): {np.round(v1_grid[0, :5], 3)} ...")
print(f"x=7 (row 6): {np.round(v1_grid[6, :5], 3)} ...")

print("\n--- v1 - v0 (positive => prefer replace) ---")
diff_vv = v1_grid - v0_grid
print(f"x=1 (row 0): {np.round(diff_vv[0, :5], 3)} ...")
print(f"x=7 (row 6): {np.round(diff_vv[6, :5], 3)} ...")

EV shape:      (7, 25)       (rows=mileage states, cols=RC grid points)
v0_grid shape: (7, 25)
v1_grid shape: (7, 25)

--- EV (integrated value function) ---
      rc_grid:   [29.8 31.3 32.8 34.3 35.7] ...
x=1 (row 0): [-109.12 -109.12 -109.12 -109.12 -109.12] ...
x=4 (row 3): [-122.652 -122.652 -122.652 -122.652 -122.652] ...
x=7 (row 6): [-128.442 -128.451 -128.453 -128.454 -128.455] ...

--- v0_grid (value of MAINTAINING) ---
x=1 (row 0): [-109.697 -109.697 -109.697 -109.697 -109.697] ...
x=7 (row 6): [-129.03  -129.031 -129.031 -129.032 -129.032] ...

--- v1_grid (value of REPLACING) ---
x=1 (row 0): [-133.492 -134.967 -136.442 -137.917 -139.392] ...
x=7 (row 6): [-133.492 -134.967 -136.442 -137.917 -139.392] ...

--- v1 - v0 (positive => prefer replace) ---
x=1 (row 0): [-23.794 -25.269 -26.744 -28.219 -29.694] ...
x=7 (row 6): [ -4.461  -5.936  -7.41   -8.885 -10.36 ] ...


## 7. Choice Probabilities via Linear Interpolation

After the inner loop, $v_0$ and $v_1$ are defined **on the RC grid**.
Observed RC values won't fall exactly on grid points, so we linearly interpolate
then apply the logit:

$$P(\text{replace} \mid x, rc) = \frac{1}{1 + \exp(v_0 - v_1)}$$

**Matrix key:** all $n$ observations are interpolated simultaneously using
vectorised index arrays — no observation-level loop.

In [71]:
def compute_choice_probs(v0_grid, v1_grid, rc_grid, x_obs, rc_obs):
    """
    Linearly interpolate v0, v1 at each observed (x, rc) and return P(replace).

    Returns
    -------
    prob_replace : ndarray (n_obs,)
    """
    N_rc   = len(rc_grid)
    rc_min = rc_grid[0]
    rc_max = rc_grid[-1]
    delta  = rc_grid[1] - rc_grid[0]

    x_obs  = np.asarray(x_obs,  dtype=int)    # 1-indexed mileage states
    rc_obs = np.asarray(rc_obs, dtype=float)

    # Clamp observed RC to grid bounds
    rc_clamped = np.clip(rc_obs, rc_min, rc_max)

    # Left grid index j such that rc_grid[j] <= rc_clamped < rc_grid[j+1]
    j = np.clip(np.floor((rc_clamped - rc_min) / delta).astype(int), 0, N_rc - 2)

    # Interpolation weight w in [0, 1]
    w = (rc_clamped - rc_grid[j]) / delta

    # Convert x to 0-indexed for numpy array access
    xi = x_obs - 1

    # Vectorised linear interpolation
    # idx_left  = (xi, j),   idx_right = (xi, j+1)
    v0 = (1 - w) * v0_grid[xi, j] + w * v0_grid[xi, j + 1]
    v1 = (1 - w) * v1_grid[xi, j] + w * v1_grid[xi, j + 1]

    # Logit: P(a=1 | x, rc) = 1 / (1 + exp(v0 - v1))
    return 1.0 / (1.0 + np.exp(v0 - v1))

### Inspect choice probabilities at warm-start values

In [72]:
prob_replace = compute_choice_probs(v0_grid, v1_grid, rc_grid, df["x"], df["rc"])

print(f"prob_replace shape: {prob_replace.shape}")
print(f"  min  = {prob_replace.min():.6f}")
print(f"  mean = {prob_replace.mean():.6f}")
print(f"  max  = {prob_replace.max():.6f}")

print("\nFirst 10 observations:")
print(pd.DataFrame({
    "x"        : df["x"].values[:10],
    "rc"       : df["rc"].values[:10],
    "a_obs"    : df["a"].values[:10],
    "P(rep)"   : np.round(prob_replace[:10], 4),
}))

prob_replace shape: (100000,)
  min  = 0.000000
  mean = 0.000003
  max  = 0.002165

First 10 observations:
   x    rc  a_obs  P(rep)
0  4  45.0      0     0.0
1  5  49.0      0     0.0
2  6  54.0      0     0.0
3  7  47.0      1     0.0
4  1  56.0      0     0.0
5  2  52.5      0     0.0
6  3  55.0      0     0.0
7  4  50.5      0     0.0
8  5  52.0      0     0.0
9  6  42.0      0     0.0


## 8. Negative Log-Likelihood — The NFXP Objective

$$\ell_{choice} = \sum_{i,t} \bigl[ a_{it} \log P(1|x_{it}, RC_t)
               + (1-a_{it}) \log P(0|x_{it}, RC_t) \bigr]$$

$$\ell_{RC} = \sum_{i} \sum_{t=2}^{T}
  \log \phi\!\left(\frac{RC_t - \rho_0 - \rho_1 RC_{t-1}}{\sigma_\rho}\right)
  - \log \sigma_\rho$$

Parameter vector: $[\theta_1,\ \theta_2,\ \rho_0,\ \rho_1,\ \log\sigma_\rho]$
— log-transforming $\sigma_\rho$ enforces positivity without box constraints.

In [73]:
# Cache numpy arrays from the dataframe once, outside the hot loop
a_obs  = df["a"].values.astype(float)
x_obs  = df["x"].values.astype(int)
rc_obs = df["rc"].values.astype(float)

def neg_log_likelihood(params):
    theta1, theta2, rho0, rho1, log_sigma_rho = params
    sigma_rho = np.exp(log_sigma_rho)   # back-transform: enforces sigma_rho > 0

    # Guard rails: reject infeasible parameter regions
    if abs(rho1) >= 0.999 or theta1 <= 0 or theta2 <= 0:
        return 1e10

    # --- Step A: Tauchen discretisation for current AR(1) parameters ---
    rc_grid, Pi = tauchen(rho0, rho1, sigma_rho, N_RC)

    # --- Step B (Inner Loop): Bellman fixed point ---
    _, v0_grid, v1_grid = solve_bellman(theta1, theta2, rc_grid, Pi)

    # --- Step C: Choice probabilities at observed (x, rc) ---
    prob_replace = compute_choice_probs(v0_grid, v1_grid, rc_grid, x_obs, rc_obs)

    # --- Step D: Choice log-likelihood (binary logit) ---
    eps_floor = 1e-12    # prevent log(0)
    ll_choice = np.sum(
        a_obs * np.log(prob_replace + eps_floor) +
        (1.0 - a_obs) * np.log(1.0 - prob_replace + eps_floor)
    )

    # --- Step E: AR(1) log-likelihood for RC_t ---
    # residuals[t] = (RC_t - rho0 - rho1*RC_{t-1}) / sigma_rho
    residuals = (rc_curr_vec - rho0 - rho1 * rc_prev_vec) / sigma_rho
    ll_rc     = np.sum(norm.logpdf(residuals)) - len(residuals) * np.log(sigma_rho)

    return -(ll_choice + ll_rc)   # return NEGATIVE LL (scipy.optimize minimises)

### Check likelihood at warm-start values

In [74]:
params_init = np.array([1.0, 1.0, rho0_ols, rho1_ols, np.log(sigma_rho_ols)])

print("--- Warm-start parameters ---")
print(f"  theta1    = {params_init[0]:.4f}")
print(f"  theta2    = {params_init[1]:.4f}")
print(f"  rho0      = {params_init[2]:.4f}")
print(f"  rho1      = {params_init[3]:.4f}")
print(f"  sigma_rho = {np.exp(params_init[4]):.4f}")
print(f"  Initial neg-LL = {neg_log_likelihood(params_init):.4f}")

--- Warm-start parameters ---
  theta1    = 1.0000
  theta2    = 1.0000
  rho0      = 17.8269
  rho1      = 0.6249
  sigma_rho = 4.6060
  Initial neg-LL = 680866.3141


## 9. Outer Loop — NFXP Optimisation

We minimise the negative log-likelihood using **Nelder-Mead** — a derivative-free method
robust to the non-smooth objective that arises from value function iteration.

**Starting values:**
- $\theta_1, \theta_2 = 1.0$ (generic positive guess)
- $\rho_0, \rho_1$ — OLS estimates from Section 4
- $\log\sigma_\rho$ — log of OLS residual std

In [75]:
result = minimize(
    neg_log_likelihood,
    params_init,
    method="Nelder-Mead",
    options=dict(
        maxiter = 5000,
        xatol   = 1e-6,
        fatol   = 1e-6,
        disp    = True,
    )
)

Optimization terminated successfully.
         Current function value: 325548.526373
         Iterations: 290
         Function evaluations: 483


## 10. Results

In [76]:
params_hat    = result.x
sigma_rho_hat = np.exp(params_hat[4])

print("=" * 47)
print("       NFXP ESTIMATION RESULTS")
print("=" * 47)
print(f"  theta1    (mileage cost coeff.)    = {params_hat[0]:.4f}")
print(f"  theta2    (replacement cost coeff) = {params_hat[1]:.4f}")
print(f"  rho0      (AR(1) intercept)        = {params_hat[2]:.4f}")
print(f"  rho1      (AR(1) persistence)      = {params_hat[3]:.4f}")
print(f"  sigma_rho (AR(1) std dev)          = {sigma_rho_hat:.4f}")
print("-" * 47)
print(f"  Log-likelihood                     = {-result.fun:.4f}")
print(f"  Convergence code: {result.status}  ({('SUCCESS' if result.success else 'check result.message')})")
print(f"  Function evaluations: {result.nfev}")
print("=" * 47)

       NFXP ESTIMATION RESULTS
  theta1    (mileage cost coeff.)    = 0.4150
  theta2    (replacement cost coeff) = 0.1578
  rho0      (AR(1) intercept)        = 17.8425
  rho1      (AR(1) persistence)      = 0.6246
  sigma_rho (AR(1) std dev)          = 4.6059
-----------------------------------------------
  Log-likelihood                     = -325548.5264
  Convergence code: 0  (SUCCESS)
  Function evaluations: 483
